# Classification — Consolidated 10-Algorithm Comparison

This notebook consolidates the five Review-1 Part-A algorithms and five Review-2 Part-B algorithms on the **same Adult Income dataset**, with the same stratified 80:20 split and leakage-safe preprocessing. It retrains all ten models so the table is reproducible and does not depend on stale CSV result files.

## Required metrics
Accuracy, weighted Precision, weighted Recall, weighted F1-score, ROC-AUC, and confusion matrices. A final model should be selected from the actual results after execution.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "Review-2" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from review2_helpers import load_adult_dataset, split_adult, build_preprocessor, RANDOM_STATE

sns.set_theme(style="whitegrid")
df = load_adult_dataset()
print("Dataset shape:", df.shape)

Dataset shape: (48842, 15)


## Define all ten models

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5, weights="uniform", n_jobs=-1),
    "Gaussian Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=RANDOM_STATE),
    "Support Vector Machine": SVC(C=1.0, kernel="rbf", probability=True, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced_subsample"),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=RANDOM_STATE),
    "Bagging (Decision Tree)": BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=12, random_state=RANDOM_STATE), n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),
    "MLP Neural Network": MLPClassifier(hidden_layer_sizes=(64,), activation="relu", max_iter=200, early_stopping=True, random_state=RANDOM_STATE),
}
print("Models:", len(models))

Models: 10


## Train and evaluate on the same split

In [4]:
X_train, X_test, y_train, y_test = split_adult(df)
results = []
fitted_pipelines = {}
confusion_matrices = {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocessor", build_preprocessor(X_train)), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    score = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else pipe.decision_function(X_test)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision_weighted": precision_score(y_test, pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_test, pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_test, pred, average="weighted", zero_division=0),
        "roc_auc": roc_auc_score(y_test, score),
    })
    fitted_pipelines[name] = pipe
    confusion_matrices[name] = confusion_matrix(y_test, pred)
comparison = pd.DataFrame(results).sort_values("f1_weighted", ascending=False).reset_index(drop=True)
display(comparison.round(4))
results_dir = PROJECT_ROOT / "results" / "review2_classification"
results_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(results_dir / "classification_all_10_models.csv", index=False)
print("Saved:", results_dir / "classification_all_10_models.csv")

,model,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc
0,Gradient Boosting,0.8676,0.8625,0.8676,0.8608,0.9214
1,Bagging (Decision Tree),0.8657,0.8602,0.8657,0.8593,0.9179
2,Random Forest,0.8557,0.8501,0.8557,0.8514,0.9046
3,MLP Neural Network,0.8555,0.8495,0.8555,0.8505,0.9077
4,Decision Tree,0.8554,0.8487,0.8554,0.8480,0.8949
5,AdaBoost,0.8569,0.8512,0.8569,0.8469,0.9070
6,Logistic Regression,0.8525,0.8458,0.8525,0.8464,0.9040
7,Support Vector Machine,0.8505,0.8434,0.8505,0.8436,0.8918
8,K-Nearest Neighbors,0.8268,0.8205,0.8268,0.8228,0.8352
9,Gaussian Naive Bayes,0.5889,0.8161,0.5889,0.6114,0.8037


Saved: /Users/venkataramana/Documents/SEMESTERS/5TH SEMESTER/MACHINE LEARNING/ML_Capstone/Review-2/results/review2_classification/classification_all_10_models.csv


## Metric Comparison

In [ ]:
plot_df = comparison.set_index("model")[["accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "roc_auc"]]
ax = plot_df.plot(kind="bar", figsize=(15, 6))
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Classification — Comparison of All Ten Algorithms")
plt.xticks(rotation=35, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Confusion Matrices for All Ten Models

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, name in zip(axes.flat, models.keys()):
    ConfusionMatrixDisplay(confusion_matrix=confusion_matrices[name], display_labels=["<=50K", ">50K"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(name, fontsize=9)
plt.tight_layout()
plt.show()

## Hyperparameter Tuning and Final Model Selection

In [ ]:
# Tune the top-performing model from the actual baseline table. Search space is deliberately compact.
best_baseline_name = comparison.iloc[0]["model"]
print("Top baseline by weighted F1:", best_baseline_name)
tuning_grids = {
    "Logistic Regression": {"model__C": [0.5, 1.0, 2.0]},
    "K-Nearest Neighbors": {"model__n_neighbors": [5, 9, 15]},
    "Gaussian Naive Bayes": {"model__var_smoothing": [1e-9, 1e-8, 1e-7]},
    "Decision Tree": {"model__max_depth": [8, 12, 16]},
    "Support Vector Machine": {"model__C": [0.5, 1.0, 2.0]},
    "Random Forest": {"model__n_estimators": [100, 200], "model__max_depth": [None, 20]},
    "AdaBoost": {"model__n_estimators": [100, 200], "model__learning_rate": [0.3, 0.7]},
    "Gradient Boosting": {"model__n_estimators": [100, 150], "model__learning_rate": [0.05, 0.1]},
    "Bagging (Decision Tree)": {"model__n_estimators": [30, 50]},
    "MLP Neural Network": {"model__hidden_layer_sizes": [(64,), (100,)], "model__activation": ["relu", "tanh"]},
}
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
tuned_search = GridSearchCV(
    Pipeline([("preprocessor", build_preprocessor(X_train)), ("model", clone(models[best_baseline_name]))]),
    tuning_grids[best_baseline_name], scoring="f1_weighted", cv=cv, n_jobs=-1, verbose=1
)
tuned_search.fit(X_train, y_train)
tuned_pipe = tuned_search.best_estimator_
tuned_pred = tuned_pipe.predict(X_test)
tuned_score = tuned_pipe.predict_proba(X_test)[:, 1] if hasattr(tuned_pipe, "predict_proba") else tuned_pipe.decision_function(X_test)
tuned_result = {
    "model": best_baseline_name + " (Tuned)",
    "accuracy": accuracy_score(y_test, tuned_pred),
    "precision_weighted": precision_score(y_test, tuned_pred, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, tuned_pred, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, tuned_pred, average="weighted", zero_division=0),
    "roc_auc": roc_auc_score(y_test, tuned_score),
}
print("Best parameters:", tuned_search.best_params_)
print("CV weighted F1:", round(tuned_search.best_score_, 4))
display(pd.DataFrame([comparison.iloc[0].to_dict(), tuned_result]).round(4))
ConfusionMatrixDisplay.from_predictions(y_test, tuned_pred, display_labels=["<=50K", ">50K"], cmap="Blues", values_format="d")
plt.title(best_baseline_name + " (Tuned) — Confusion Matrix")
plt.tight_layout()
plt.show()
pd.DataFrame([tuned_result]).to_csv(results_dir / "best_tuned_model.csv", index=False)

## Final Interpretation — Complete After Running
Use the displayed table and plots to identify the best model. Explain trade-offs among accuracy, weighted F1, and ROC-AUC, and state whether tuning improved at least one metric.